# SODA Defence — Thesis Figures & Timing
Generates all thesis-ready visualisations and a stage-by-stage latency breakdown for SODA.

**Final SODA params (CARLA-GeAR):** down=160, z=2.0, dilate=2, morph=5, w_sal=1.0, w_edge=0.8, w_sat=0.4
**Final SODA params (Isaac Sim):**  down=160, z=2.5, dilate=0, morph=3, w_sal=1.0, w_edge=0.3, w_sat=0.0


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics opencv-python-headless scikit-learn pandas pillow

In [ ]:
import os, time, shutil
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from pathlib import Path
from PIL import Image
from ultralytics import YOLO

THESIS_FIGS = Path("/content/soda_thesis_figures")
THESIS_FIGS.mkdir(exist_ok=True)

plt.rcParams.update({
    'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11,
    'legend.fontsize': 10, 'figure.dpi': 150,
    'savefig.dpi': 300, 'savefig.bbox': 'tight'
})

def save_fig(name):
    p = THESIS_FIGS / f"{name}.png"
    plt.savefig(p, bbox_inches='tight', dpi=300)
    print(f"  ✅ Saved: {p}")

print("Setup complete. Figures →", THESIS_FIGS)

## 2. Paths & Model — EDIT THESE

In [ ]:
MODEL_PATH       = "/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/best.pt"
PATCHED_IMG_ROOT = "/content/std_patched/images/test"   # has billboard01..09 subfolders
CLEAN_IMG_ROOT   = "/content/std_clean/images/test"
DEMO_SCENARIO    = "billboard03"                         # scenario used for demo figures
DEMO_IMG_IDX     = 0                                     # which image in that scenario

# Final SODA hyperparameters
SODA_PARAMS_CARLA = dict(
    down=160, z=2.0, dilate=2, morph=5,
    w_sal=1.0, w_edge=0.8, w_sat=0.4,
    min_area_frac=0.001, max_area_frac=0.25,
    inpaint_radius=3, feather=2.0
)

SODA_PARAMS_ISAAC = dict(
    down=160, z=2.5, dilate=0, morph=3,
    w_sal=1.0, w_edge=0.3, w_sat=0.0,
    min_area_frac=0.001, max_area_frac=0.08,
    inpaint_radius=3, feather=2.0
)

model = YOLO(MODEL_PATH)
device = next(model.model.parameters()).device
model.model.eval()
print("Model loaded. Device:", device)

## 3. SODA Core Functions (paper-identical)

In [ ]:
def _spectral_residual_saliency(gray01):
    """Hou-Zhang spectral residual saliency. Input gray in [0,1], output in [0,1]."""
    eps = 1e-8
    F = np.fft.fft2(gray01)
    A = np.abs(F) + eps
    P = np.angle(F)
    logA = np.log(A)
    logA_blur = cv2.GaussianBlur(logA.astype(np.float32), (0, 0), 3.0)
    R = logA - logA_blur
    S = np.exp(R + 1j * P)
    s = np.fft.ifft2(S)
    sal = (np.abs(s) ** 2).astype(np.float32)
    sal = (sal - sal.min()) / (sal.max() - sal.min() + eps)
    return cv2.GaussianBlur(sal, (0, 0), 2.0)


def soda_score_maps(img_rgb_uint8, params):
    """Returns (sal, edge, sat, fused_score) all at downsampled resolution."""
    down = int(params.get('down', 160))
    w_sal = float(params.get('w_sal', 1.0))
    w_edge = float(params.get('w_edge', 0.8))
    w_sat = float(params.get('w_sat', 0.4))

    H, W = img_rgb_uint8.shape[:2]
    scale = down / max(H, W)
    nh, nw = int(round(H * scale)), int(round(W * scale))
    small = cv2.resize(img_rgb_uint8, (nw, nh), interpolation=cv2.INTER_AREA)

    gray = cv2.cvtColor(small, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    hsv  = cv2.cvtColor(small, cv2.COLOR_RGB2HSV)
    sat  = hsv[:, :, 1].astype(np.float32) / 255.0

    sal  = _spectral_residual_saliency(gray)
    lap  = cv2.Laplacian((gray * 255).astype(np.uint8), cv2.CV_32F, ksize=3)
    edge = np.abs(lap).astype(np.float32)
    edge = (edge - edge.min()) / (edge.max() - edge.min() + 1e-8)

    score = w_sal * sal + w_edge * edge + w_sat * sat
    return sal, edge, sat, score


def soda_defend(img_rgb_uint8, params):
    """Full SODA pipeline. Returns (defended_uint8, mask_fullres, score_fullres)."""
    down  = int(params.get('down', 160))
    z     = float(params.get('z', 2.0))
    dilate = int(params.get('dilate', 2))
    morph  = int(params.get('morph', 5))
    w_sal  = float(params.get('w_sal', 1.0))
    w_edge = float(params.get('w_edge', 0.8))
    w_sat  = float(params.get('w_sat', 0.4))
    min_af = float(params.get('min_area_frac', 0.001))
    max_af = float(params.get('max_area_frac', 0.25))
    ir     = int(params.get('inpaint_radius', 3))
    feather = float(params.get('feather', 2.0))

    H, W = img_rgb_uint8.shape[:2]
    scale = down / max(H, W)
    nh, nw = int(round(H * scale)), int(round(W * scale))
    small = cv2.resize(img_rgb_uint8, (nw, nh), interpolation=cv2.INTER_AREA)

    gray = cv2.cvtColor(small, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    hsv  = cv2.cvtColor(small, cv2.COLOR_RGB2HSV)
    sat  = hsv[:, :, 1].astype(np.float32) / 255.0

    sal  = _spectral_residual_saliency(gray)
    lap  = cv2.Laplacian((gray * 255).astype(np.uint8), cv2.CV_32F, ksize=3)
    edge = np.abs(lap).astype(np.float32)
    edge = (edge - edge.min()) / (edge.max() - edge.min() + 1e-8)
    score = w_sal * sal + w_edge * edge + w_sat * sat

    thr  = score.mean() + z * (score.std() + 1e-8)
    mask = (score > thr).astype(np.uint8)

    if morph > 0:
        k = np.ones((morph, morph), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k)
    if dilate > 0:
        k2 = np.ones((max(3, morph), max(3, morph)), np.uint8)
        mask = cv2.dilate(mask, k2, iterations=dilate)

    # area gate
    frac = mask.sum() / (nh * nw + 1e-8)
    if frac < min_af or frac > max_af:
        mask_full = np.zeros((H, W), dtype=np.uint8)
        score_full = cv2.resize(score, (W, H), interpolation=cv2.INTER_LINEAR)
        return img_rgb_uint8.copy(), mask_full, score_full

    # connected components — keep largest
    num, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num > 1:
        areas = stats[1:, cv2.CC_STAT_AREA]
        best  = int(np.argmax(areas)) + 1
        mask  = (labels == best).astype(np.uint8)

    # upsample mask
    mask_full  = cv2.resize(mask, (W, H), interpolation=cv2.INTER_NEAREST)
    score_full = cv2.resize(score, (W, H), interpolation=cv2.INTER_LINEAR)

    # feathered inpaint
    if feather > 0:
        mf = cv2.GaussianBlur(mask_full.astype(np.float32), (0, 0), feather)
        mf = np.clip(mf, 0, 1)
    else:
        mf = mask_full.astype(np.float32)

    img_bgr  = cv2.cvtColor(img_rgb_uint8, cv2.COLOR_RGB2BGR)
    inpainted = cv2.inpaint(img_bgr, mask_full, ir, cv2.INPAINT_TELEA)
    inpainted_rgb = cv2.cvtColor(inpainted, cv2.COLOR_BGR2RGB)

    out = (img_rgb_uint8.astype(np.float32) * (1 - mf[..., None]) +
           inpainted_rgb.astype(np.float32) * mf[..., None])
    return np.clip(out, 0, 255).astype(np.uint8), mask_full, score_full


print("SODA functions defined.")

In [ ]:
MODEL_PATH       = "/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/best.pt"

# ── Check which layout you actually have ─────────────────────────────────────
# Layout A (flat):   PATCHED_IMG_ROOT/rgb_0001.png
# Layout B (nested): PATCHED_IMG_ROOT/billboard03/rgb_0001.png

# Try layout A first — flat folder with all images
PATCHED_IMG_ROOT = "/content/drive/MyDrive/Patched Datasets/patched_dataset1/images/test"
CLEAN_IMG_ROOT   = "/content/drive/MyDrive/2dod/2dod/images/test"
DEMO_SCENARIO    = ""    # leave empty for flat layout
DEMO_IMG_IDX     = 0

# Final SODA hyperparameters
SODA_PARAMS_CARLA = dict(
    down=160, z=2.0, dilate=2, morph=5,
    w_sal=1.0, w_edge=0.8, w_sat=0.4,
    min_area_frac=0.001, max_area_frac=0.25,
    inpaint_radius=3, feather=2.0
)

SODA_PARAMS_ISAAC = dict(
    down=160, z=2.5, dilate=0, morph=3,
    w_sal=1.0, w_edge=0.3, w_sat=0.0,
    min_area_frac=0.001, max_area_frac=0.08,
    inpaint_radius=3, feather=2.0
)

model = YOLO(MODEL_PATH)
device = next(model.model.parameters()).device
model.model.eval()
print("Model loaded. Device:", device)

In [ ]:
from ultralytics import YOLO
import torch

MODEL_PATH = "/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/best.pt"

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = YOLO(MODEL_PATH)
model.to(device)
model.model.eval()

print("Model loaded. Device:", next(model.model.parameters()).device)

## 4. Stage-by-Stage Timing Analysis

In [ ]:
import glob

# Load demo image
demo_dir  = Path(PATCHED_IMG_ROOT) / DEMO_SCENARIO
img_paths = sorted(list(demo_dir.glob('*.png')) + list(demo_dir.glob('*.jpg')))
demo_img  = np.array(Image.open(img_paths[DEMO_IMG_IDX]).convert('RGB'))
print(f"Demo image: {img_paths[DEMO_IMG_IDX].name}  shape={demo_img.shape}")

params = SODA_PARAMS_CARLA
N = 20  # timing repetitions

stages = {
    'Stage 1: Downsample + Score Maps': [],
    'Stage 2: Threshold + Morphology':  [],
    'Stage 3: Inpaint + Blend':         [],
    'Total':                            []
}

for _ in range(N):
    img = demo_img.copy()
    H, W = img.shape[:2]
    down  = int(params['down'])
    scale = down / max(H, W)
    nh, nw = int(round(H * scale)), int(round(W * scale))

    # --- Stage 1: downsample + compute score maps ---
    t0 = time.perf_counter()
    small = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    gray  = cv2.cvtColor(small, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    hsv   = cv2.cvtColor(small, cv2.COLOR_RGB2HSV)
    sat   = hsv[:, :, 1].astype(np.float32) / 255.0
    sal   = _spectral_residual_saliency(gray)
    lap   = cv2.Laplacian((gray * 255).astype(np.uint8), cv2.CV_32F, ksize=3)
    edge  = np.abs(lap).astype(np.float32)
    edge  = (edge - edge.min()) / (edge.max() - edge.min() + 1e-8)
    score = params['w_sal'] * sal + params['w_edge'] * edge + params['w_sat'] * sat
    t1 = time.perf_counter()

    # --- Stage 2: threshold + morphology + CC ---
    thr  = score.mean() + params['z'] * (score.std() + 1e-8)
    mask = (score > thr).astype(np.uint8)
    if params['morph'] > 0:
        k = np.ones((params['morph'], params['morph']), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k)
    if params['dilate'] > 0:
        k2 = np.ones((max(3, params['morph']), max(3, params['morph'])), np.uint8)
        mask = cv2.dilate(mask, k2, iterations=params['dilate'])
    num, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num > 1:
        best = int(np.argmax(stats[1:, cv2.CC_STAT_AREA])) + 1
        mask = (labels == best).astype(np.uint8)
    mask_full = cv2.resize(mask, (W, H), interpolation=cv2.INTER_NEAREST)
    t2 = time.perf_counter()

    # --- Stage 3: feather + inpaint + blend ---
    mf = cv2.GaussianBlur(mask_full.astype(np.float32), (0, 0), params['feather'])
    mf = np.clip(mf, 0, 1)
    img_bgr   = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    inpainted = cv2.inpaint(img_bgr, mask_full, params['inpaint_radius'], cv2.INPAINT_TELEA)
    inpainted_rgb = cv2.cvtColor(inpainted, cv2.COLOR_BGR2RGB)
    out = (img.astype(np.float32) * (1 - mf[..., None]) +
           inpainted_rgb.astype(np.float32) * mf[..., None])
    np.clip(out, 0, 255).astype(np.uint8)
    t3 = time.perf_counter()

    stages['Stage 1: Downsample + Score Maps'].append((t1 - t0) * 1000)
    stages['Stage 2: Threshold + Morphology'].append((t2 - t1) * 1000)
    stages['Stage 3: Inpaint + Blend'].append((t3 - t2) * 1000)
    stages['Total'].append((t3 - t0) * 1000)

print(f"{'='*60}")
print(f"{'SODA Stage Timing':<40} {'Mean':>8} {'Std':>8} {'%':>6}")
print(f"{'='*60}")
rows = []
total_mean = np.mean(stages['Total'])
for stage, times in stages.items():
    m, s = np.mean(times), np.std(times)
    pct = m / total_mean * 100
    print(f"{stage:<40} {m:>8.1f} {s:>8.1f} {pct:>5.1f}%")
    rows.append({'Stage': stage, 'Mean (ms)': round(m, 1), 'Std (ms)': round(s, 1), '% of Total': round(pct, 1)})
print(f"{'='*60}")

df_timing = pd.DataFrame(rows)
df_timing.to_csv(THESIS_FIGS / 'soda_stage_timing.csv', index=False)
print('\n✅ Timing CSV saved')

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
stage_labels = ['Score Maps', 'Threshold\n& Morphology', 'Inpaint\n& Blend']
means = [np.mean(stages[k]) for k in list(stages.keys())[:3]]
stds  = [np.std(stages[k])  for k in list(stages.keys())[:3]]
colors = ['#4C72B0', '#DD8452', '#55A868']
bars = ax.bar(stage_labels, means, yerr=stds, capsize=5, color=colors)
ax.set_ylabel('Time (ms)')
ax.set_title(f'SODA Stage Latency  (n={N} runs, {demo_img.shape[1]}×{demo_img.shape[0]})')
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(stds)*0.1,
            f'{m:.1f} ms', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
save_fig('soda_stage_timing_bar')
plt.show()
print(f'\nBottleneck: Stage 3 (Inpaint) is expected to dominate.')
print(f'Total mean: {total_mean:.1f} ms/image')

## 5. SODA 6-Panel Pipeline Figure (Thesis-Ready)

In [ ]:
params = SODA_PARAMS_CARLA

# Compute all maps
sal, edge, sat_map, score = soda_score_maps(demo_img, params)
defended, mask_full, score_full = soda_defend(demo_img, params)

# Upsample maps for display
H, W = demo_img.shape[:2]
sal_up   = cv2.resize(sal,   (W, H), interpolation=cv2.INTER_LINEAR)
edge_up  = cv2.resize(edge,  (W, H), interpolation=cv2.INTER_LINEAR)
sat_up   = cv2.resize(sat_map, (W, H), interpolation=cv2.INTER_LINEAR)
score_up = cv2.resize(score, (W, H), interpolation=cv2.INTER_LINEAR)

# ── 6-panel figure ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

axes[0].imshow(demo_img);                              axes[0].set_title('(a) Patched input')
axes[1].imshow(sal_up, cmap='hot');                    axes[1].set_title('(b) Spectral residual saliency S_sal')
axes[2].imshow(edge_up, cmap='hot');                   axes[2].set_title('(c) Laplacian edge density S_edge')
axes[3].imshow(sat_up, cmap='hot');                    axes[3].set_title('(d) Colour saturation S_sat')
axes[4].imshow(score_up, cmap='hot');                  axes[4].set_title('(e) Fused patchness score S')
# Show mask as outline on defended image
vis_mask = defended.copy()
contours, _ = cv2.findContours(mask_full, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cv2.drawContours(vis_mask, contours, -1, (255, 0, 0), 3)
axes[5].imshow(vis_mask);                              axes[5].set_title('(f) SODA output (mask outline in red)')

for ax in axes: ax.axis('off')
plt.suptitle(
    f'SODA Pipeline  (w_sal={params["w_sal"]}, w_edge={params["w_edge"]}, '
    f'w_sat={params["w_sat"]}, z={params["z"]}, down={params["down"]})',
    fontsize=13, y=1.01
)
plt.tight_layout()
save_fig('soda_pipeline_6panel')
plt.show()

# Save individual panels
panels = [
    (demo_img,   'soda_p1_input',   None),
    (sal_up,     'soda_p2_saliency','hot'),
    (edge_up,    'soda_p3_edge',    'hot'),
    (sat_up,     'soda_p4_sat',     'hot'),
    (score_up,   'soda_p5_score',   'hot'),
    (vis_mask,   'soda_p6_output',  None),
]
for img, name, cmap in panels:
    fig2, ax2 = plt.subplots(figsize=(5, 4))
    ax2.imshow(img, cmap=cmap); ax2.axis('off')
    plt.tight_layout(pad=0)
    save_fig(name)
    plt.close()

## 6. SODA Mask Localisation — Clean vs Patched Overlay

In [ ]:
# Load matching clean image
clean_dir   = Path(CLEAN_IMG_ROOT) / DEMO_SCENARIO
clean_paths = sorted(list(clean_dir.glob('*.png')) + list(clean_dir.glob('*.jpg')))

if len(clean_paths) > DEMO_IMG_IDX:
    clean_img = np.array(Image.open(clean_paths[DEMO_IMG_IDX]).convert('RGB'))

    _, mask_clean, score_clean = soda_defend(clean_img, SODA_PARAMS_CARLA)
    _, mask_patched, score_patched = soda_defend(demo_img, SODA_PARAMS_CARLA)

    # Overlay: show where clean and patched masks differ
    diff_mask = np.zeros((*mask_clean.shape, 3), dtype=np.uint8)
    diff_mask[mask_patched > 0] = [255, 0, 0]    # patched-only: red
    diff_mask[mask_clean > 0]   = [0, 200, 0]    # clean-only: green
    # Overlap: yellow
    both = (mask_patched > 0) & (mask_clean > 0)
    diff_mask[both] = [255, 200, 0]

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(clean_img);   axes[0].set_title('Clean input')
    axes[1].imshow(demo_img);    axes[1].set_title('Patched input')

    clean_vis = clean_img.copy()
    c_ctrs, _ = cv2.findContours(mask_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(clean_vis, c_ctrs, -1, (0, 200, 0), 3)
    axes[2].imshow(clean_vis);   axes[2].set_title('Clean mask (green)')

    patch_vis = demo_img.copy()
    p_ctrs, _ = cv2.findContours(mask_patched, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(patch_vis, p_ctrs, -1, (255, 0, 0), 3)
    axes[3].imshow(patch_vis);   axes[3].set_title('Patched mask (red)')

    for ax in axes: ax.axis('off')
    plt.suptitle('SODA Mask: Clean vs Patched — patch causes new/shifted localisation', fontsize=12, y=1.01)
    plt.tight_layout()
    save_fig('soda_clean_vs_patched_mask')
    plt.show()

    iou = float((both.sum()) / ((mask_patched | mask_clean).sum() + 1e-8))
    print(f'IoU(clean mask, patched mask): {iou:.3f}')
    print('Low IoU = mask moves with patch = good localisation sensitivity')
else:
    print('Clean images not found at:', clean_dir, '— adjust CLEAN_IMG_ROOT')

## 7. YOLO Detection Before and After SODA

In [ ]:
import torch

def yolo_vis(img_rgb, conf=0.3):
    t = torch.from_numpy(img_rgb).permute(2, 0, 1).float().to(device) / 255.0
    with torch.no_grad():
        result = model(t.unsqueeze(0), verbose=False)[0]
    vis = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
    n   = 0 if result.boxes is None else len(result.boxes)
    return vis, n

vis_patched,  n_p = yolo_vis(demo_img)
vis_defended, n_d = yolo_vis(defended)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(vis_patched);  axes[0].set_title(f'Patched — {n_p} detections')
axes[1].imshow(vis_defended); axes[1].set_title(f'SODA Defended — {n_d} detections')
for ax in axes: ax.axis('off')
plt.suptitle('YOLOv8n: Patched vs SODA Defended', fontsize=13, y=1.01)
plt.tight_layout()
save_fig('soda_yolo_before_after')
plt.show()
print(f'Detections: Patched={n_p}  →  SODA Defended={n_d}')

## 8. Pixel Difference Heatmap (Where SODA Changed the Image)

In [ ]:
diff = np.abs(demo_img.astype(np.int16) - defended.astype(np.int16)).mean(axis=2).astype(np.float32)
diff_norm = (255 * diff / (diff.max() + 1e-6)).astype(np.uint8)
heat_bgr  = cv2.applyColorMap(diff_norm, cv2.COLORMAP_JET)
heat_rgb  = cv2.cvtColor(heat_bgr, cv2.COLOR_BGR2RGB)

# Overlay on patched image
overlay_bgr = cv2.addWeighted(
    cv2.cvtColor(demo_img, cv2.COLOR_RGB2BGR), 0.6, heat_bgr, 0.4, 0
)
overlay_rgb = cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(demo_img);    axes[0].set_title('Patched input')
axes[1].imshow(heat_rgb);    axes[1].set_title('Pixel change heatmap')
axes[2].imshow(overlay_rgb); axes[2].set_title('Overlay')
for ax in axes: ax.axis('off')
plt.suptitle('SODA: Spatial extent of image modification', fontsize=12, y=1.01)
plt.tight_layout()
save_fig('soda_pixel_diff_heatmap')
plt.show()

changed_pct = float((diff > 2).mean()) * 100
print(f'Pixels changed (|diff|>2): {changed_pct:.1f}%')
print(f'Mean absolute change:      {diff.mean():.2f}')
print(f'Max absolute change:       {diff.max():.0f}')

## 9. SODA vs LGS — Score Map Comparison on Same Image

In [ ]:
# LGS mask for comparison
def lgs_mask(img_rgb_uint8, sigma=7.0, threshold=0.10, morph=3):
    img = img_rgb_uint8.astype(np.float32) / 255.0
    gray = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    hf = np.abs(cv2.Laplacian(gray, cv2.CV_32F, ksize=3))
    hf_norm = hf / (hf.max() + 1e-8)
    mask = (hf_norm > threshold).astype(np.uint8)
    if morph > 0:
        k = np.ones((morph, morph), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    return hf_norm, mask

hf_norm, lgs_m = lgs_mask(demo_img)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0, 0].imshow(demo_img);                axes[0, 0].set_title('Input')
axes[0, 1].imshow(hf_norm, cmap='hot');     axes[0, 1].set_title('LGS: Laplacian HF map')
lgs_vis = demo_img.copy()
lgs_ctrs, _ = cv2.findContours(lgs_m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cv2.drawContours(lgs_vis, lgs_ctrs, -1, (0, 200, 0), 2)
axes[0, 2].imshow(lgs_vis);                 axes[0, 2].set_title('LGS: mask (green)')

axes[1, 0].imshow(demo_img);                axes[1, 0].set_title('Input')
axes[1, 1].imshow(score_up, cmap='hot');    axes[1, 1].set_title('SODA: fused patchness score')
soda_vis = demo_img.copy()
s_ctrs, _ = cv2.findContours(mask_full, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cv2.drawContours(soda_vis, s_ctrs, -1, (255, 0, 0), 2)
axes[1, 2].imshow(soda_vis);                axes[1, 2].set_title('SODA: mask (red)')

for ax in axes.flatten(): ax.axis('off')
plt.suptitle('LGS vs SODA: localisation mechanism comparison', fontsize=13, y=1.01)
plt.tight_layout()
save_fig('soda_vs_lgs_mask_comparison')
plt.show()

## 10. Multi-Scenario SODA Localisation Gallery

In [ ]:
SCENARIOS = [f'billboard{i:02d}' for i in range(1, 10)]
params = SODA_PARAMS_CARLA

fig, axes = plt.subplots(3, 3, figsize=(18, 18))
axes = axes.flatten()

for idx, scen in enumerate(SCENARIOS):
    scen_dir  = Path(PATCHED_IMG_ROOT) / scen
    imgs = sorted(list(scen_dir.glob('*.png')) + list(scen_dir.glob('*.jpg')))
    if not imgs:
        axes[idx].axis('off'); axes[idx].set_title(f'{scen} — no images')
        continue
    img = np.array(Image.open(imgs[0]).convert('RGB'))
    defended_s, mask_s, _ = soda_defend(img, params)
    vis = img.copy()
    ctrs, _ = cv2.findContours(mask_s, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(vis, ctrs, -1, (255, 0, 0), 3)
    axes[idx].imshow(vis)
    area_pct = float(mask_s.mean()) * 100
    axes[idx].set_title(f'{scen}  mask={area_pct:.1f}%', fontsize=10)
    axes[idx].axis('off')

plt.suptitle('SODA Localisation Across All Billboard Scenarios (mask outline in red)', fontsize=13, y=1.01)
plt.tight_layout()
save_fig('soda_localization_gallery_all_scenarios')
plt.show()

## 11. SODA Real-Time Feasibility

In [ ]:
# Time full SODA pipeline per image (N images from demo scenario)
N_TIME = min(30, len(img_paths))
soda_latencies = []
for p in img_paths[:N_TIME]:
    img = np.array(Image.open(p).convert('RGB'))
    t0  = time.perf_counter()
    soda_defend(img, SODA_PARAMS_CARLA)
    soda_latencies.append((time.perf_counter() - t0) * 1000)

DETECTOR_MS = 45.2   # YOLOv8n full-pipeline E2E from thesis
SODA_MS     = np.mean(soda_latencies)
COMBINED_MS = DETECTOR_MS + SODA_MS

print('=' * 55)
print('SODA REAL-TIME FEASIBILITY')
print('=' * 55)
print(f'SODA mean latency:        {SODA_MS:.1f} ms  (std={np.std(soda_latencies):.1f})')
print(f'SODA p95 latency:         {np.percentile(soda_latencies,95):.1f} ms')
print(f'Detector E2E (no defence):{DETECTOR_MS:.1f} ms  ({1000/DETECTOR_MS:.1f} FPS)')
print(f'Combined E2E (SODA):      {COMBINED_MS:.1f} ms  ({1000/COMBINED_MS:.1f} FPS)')
print(f'30 FPS budget:            {1000/30:.1f} ms')
rt = '✅ REAL-TIME' if COMBINED_MS < 1000/30 else '❌ BELOW 30 FPS'
print(f'Assessment:               {rt}')
print('=' * 55)

pd.DataFrame([{
    'Defence': 'SODA',
    'Mean latency (ms)': round(SODA_MS, 1),
    'p95 (ms)': round(np.percentile(soda_latencies, 95), 1),
    'Detector E2E (ms)': DETECTOR_MS,
    'Combined E2E (ms)': round(COMBINED_MS, 1),
    'Combined FPS': round(1000/COMBINED_MS, 1),
    'Real-time': rt
}]).to_csv(THESIS_FIGS / 'soda_realtime_feasibility.csv', index=False)
print('✅ Feasibility CSV saved')

## 12. Summary: All Saved Figures

In [ ]:
saved = sorted(THESIS_FIGS.glob('*'))
print(f'\n{"="*55}')
print(f'THESIS FIGURES SAVED TO: {THESIS_FIGS}')
print(f'{"="*55}')
for f in saved:
    print(f'  {f.name:<50} {f.stat().st_size/1024:>7.1f} KB')
print(f'{"="*55}')
print(f'Total: {len(saved)} files')

shutil.make_archive('/content/soda_thesis_figures', 'zip', THESIS_FIGS)
print('\n✅ Zipped to /content/soda_thesis_figures.zip')